# Binding-rate search limits: a controlled extension of V5

**Question:** do the calculated binding rates and tumour–healthy-cell trade-offs depend on the allowed search limits?

T1 and T2 remain generic. Keep V5's biology, three synthetic input variants, dose, cell-kill equation and healthy-cell constraints unchanged. Compare longer searches across three declared domains. This is a numerical investigation of affinity selection, not a source of experimentally established limits for generic targets.

The notebook contains the model and analysis. Saved search records accompany it in `results_boundary_review/`. `REUSE_SAVED_RUNS=True` displays the completed study; set it to `False` to repeat all nine searches and tight integrations. Change it to `False` after any model or input change. New full runs can take tens of minutes. V5 is not modified.


## 1. Unchanged V5 model
The following equations and biological inputs are copied from V5. All new work concerns the optimisation domain and its assessment.

In [ ]:
import matplotlib
matplotlib.use('Agg')
from dataclasses import dataclass, replace, asdict
from pathlib import Path
import itertools, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from platypus import Problem, Real, Solution, InjectedPopulation, NSGAII
from IPython.display import display, Markdown

# Hours; solution and intracellular concentrations in uM; amounts in umol;
# surface/endosomal states in units of 1e-10 mol/dm2.
F = np.array([.0003, .79, .10, .10, .01])
NAMES = ['T', 'H0', 'H1', 'H2', 'H12']
NCELL = 7043996587630.85
VI = 2.689720494498626e-13
AREA = 3.141592653589793e-8
CAP = NCELL * F * AREA * 1e-4  # umol per scaled surface unit
CELLVOL = NCELL * F * VI     # L
GAIN = AREA * 1e-4 / VI      # uM per scaled surface unit
VC, VP, QA = 11.3, 41.5, 1.05
CLA = 0.015540799881601949
BOUNDS = [(2.,7.),(-5.,2.),(2.,7.),(-5.,2.)]
LIMIT = .0095
RATES = ['kon1','koff1','kon2','koff2']

@dataclass(frozen=True)
class Scenario:
    rho: tuple = (1., 1.)  # multiples of 1e-10 mol/dm2
    k0: tuple = (2., 2.)
    ki: tuple = (2., 2., 2.)
    half: float = np.inf
    permeability: float = 0.  # PS / VI, h^-1; bidirectional, K=fu=1
    payload_volume: float = 52.8  # one WELL-MIXED extracellular pool, exploratory
    payload_clearance: float = 5.28  # L/h; clearance rate 0.1 h^-1
    metabolism: float = 0.
    tau: float = 0.  # 0: instantaneous internal degradation and release
    recycle: float = 0.
    process_share: float = .5  # remaining exit split between processing/degradation
    closure: str = 'symmetric'
    surface_factor: float = 1.
    dose: float = 4.  # umol
    target_densities: tuple = ()  # optional 5 x 2 matrix, multiples of 1e-10 mol/dm2
    fractions: tuple = (.0003,.79,.10,.10,.01)
    total_cells: float = NCELL
    intracellular_volume: float = VI
    cell_area: float = AREA
    antibody_vc: float = VC
    antibody_vp: float = VP
    antibody_q: float = QA
    antibody_clearance: float = CLA
    infusion_h: float = 2.
    horizon_h: float = 24.
    recycling_rates: tuple = ()
    processing_rates: tuple = ()
    degradation_rates: tuple = ()


def density(s):
    if s.target_densities:
        return np.asarray(s.target_densities, dtype=float)
    return np.array([[1,1],[0,0],[.25,0],[0,.25],[.25,.25]]) * s.rho


def make_rhs(theta, s, on):
    CAP=s.total_cells*np.array(s.fractions)*s.cell_area*1e-4
    CELLVOL=s.total_cells*np.array(s.fractions)*s.intracellular_volume
    GAIN=s.cell_area*1e-4/s.intracellular_volume
    VC,VP,QA,CLA=s.antibody_vc,s.antibody_vp,s.antibody_q,s.antibody_clearance
    a1,b1,a2,b2 = 10.**np.asarray(theta) * 3600
    l = 2.4e13 * 1e-10 * 3600 * s.surface_factor
    corr = np.sqrt(a2/a1) if s.closure == 'symmetric' else 1.
    q1,q2 = b1*corr,b2/corr
    kd = np.log(2)/s.half
    rho,k0,ki = density(s), np.array(s.k0), np.array(s.ki)
    kr = s.recycle/s.tau if s.tau else 0.
    kp = (1-s.recycle)*s.process_share/s.tau if s.tau else 0.
    kg = (1-s.recycle)*(1-s.process_share)/s.tau if s.tau else 0.
    if s.recycling_rates:
        kr,kp,kg=map(np.asarray,(s.recycling_rates,s.processing_rates,s.degradation_rates))
    def fun(t,y):
        # 0:4 Ic,Ip,Nc,Np; 4 extracellular payload; 5 Ab lost; 6 payload lost.
        # per cell: R(2), intact S(3), naked S(3), intact E(3), naked E(3),
        # P instantaneous, A cumulative productive targeted release (both uM).
        z=y[7:].reshape(5,16); dz=np.zeros_like(z); out=np.zeros_like(y)
        R=z[:,:2]; S=z[:,2:8].reshape(5,2,3); E=z[:,8:14].reshape(5,2,3)
        dS=dz[:,2:8].reshape(5,2,3); dE=dz[:,8:14].reshape(5,2,3)
        dz[:,:2]=k0*(rho-R)
        for x,c in enumerate([y[0],y[2]]):
            b=S[:,x,:]
            v1=a1*c*1e-6*R[:,0]-b1*b[:,0]
            v2=a2*c*1e-6*R[:,1]-b2*b[:,1]
            w1=l*b[:,1]*R[:,0]-q1*b[:,2]
            w2=l*b[:,0]*R[:,1]-q2*b[:,2]
            dz[:,0]-=v1+w1; dz[:,1]-=v2+w2
            dS[:,x,:]=np.column_stack([v1-w2,v2-w1,w1+w2])-ki*b
            cix=2*x; pix=cix+1
            flow=QA*(y[cix]-y[pix])
            out[cix]=(-CLA*y[cix]-flow-CAP@(v1+v2))/VC
            out[pix]=flow/VP
        shed=kd*S[:,0,:]
        dS[:,0,:]-=shed; dS[:,1,:]+=shed
        out[0]-=kd*y[0]; out[1]-=kd*y[1]
        out[2]+=kd*y[0]; out[3]+=kd*y[1]
        out[0]+=(s.dose/s.infusion_h/VC if on else 0.)
        if s.tau or s.recycling_rates:
            dS[:]+=kr*E
            dE[:,0,:]=ki*S[:,0,:]-(kr+kp+kg)*E[:,0,:]
            dE[:,1,:]=ki*S[:,1,:]+kp*E[:,0,:]-(kr+kg)*E[:,1,:]
            release=((kp+kg)*E[:,0,:]).sum(axis=1)
            ab_loss=(kg*E.sum(axis=1)).sum(axis=1)
        else:
            release=(ki*S[:,0,:]).sum(axis=1)
            ab_loss=(ki*S.sum(axis=1)).sum(axis=1)
        production=GAIN*release
        exchange=s.permeability*(y[4]-z[:,14])
        dz[:,14]=production+exchange-s.metabolism*z[:,14]
        dz[:,15]=production
        # Both antibody compartments release to the explicitly pooled extracellular
        # payload volume; this is a lumping assumption, not central-only transport.
        generated=kd*(VC*y[0]+VP*y[1])+CAP@shed.sum(axis=1)
        out[4]=(generated-s.payload_clearance*y[4]-CELLVOL@exchange)/s.payload_volume
        out[5]=CLA*(y[0]+y[2])+CAP@ab_loss
        out[6]=CLA*y[0]+s.payload_clearance*y[4]+s.metabolism*(CELLVOL@z[:,14])
        out[7:]=dz.ravel()
        return out
    return fun



@dataclass(frozen=True)
class KillModel:
    c50_uM: tuple = (4.,4.,4.,4.,4.)
    maximum_rate_h: tuple = (.08,.08,.08,.08,.08)

# Explicit rates replace tau/branching shortcuts in the combined model.
BIOLOGY=Scenario(ki=(.2,4.,2.), half=96., permeability=.1, metabolism=.01,
                 recycling_rates=(.6,.6,.6), processing_rates=(.7,.7,.7),
                 degradation_rates=(.7,.7,.7))
PD=KillModel()
INPUT_SET=[('Nominal',BIOLOGY,PD),
 ('Variant A',replace(BIOLOGY,ki=(.16,3.2,1.6),half=80.,permeability=.12),
  replace(PD,c50_uM=(4.8,3.2,3.2,3.2,3.2))),
 ('Variant B',replace(BIOLOGY,ki=(.24,4.8,2.4),half=120.,permeability=.08),
  replace(PD,c50_uM=(3.2,4.8,4.8,4.8,4.8)))]
GLOBAL_LIMIT=.0095
GROUP_LIMITS=np.repeat(.10,4)
SEEDS=(8407,2026)
GLOBAL_EVALUATIONS=256
LOCAL_ITERATIONS=40


def check_inputs(s,p):
    assert s.closure == "symmetric", "This method retains V1 symmetric surface closure"
    assert np.asarray(density(s)).shape==(5,2) and np.all(density(s)>=0)
    assert len(s.fractions)==5 and np.all(np.array(s.fractions)>=0)
    assert np.isclose(sum(s.fractions[1:]),1.)
    assert 0<s.infusion_h<s.horizon_h and s.dose>=0
    assert min(s.total_cells,s.intracellular_volume,s.cell_area,s.antibody_vc,s.antibody_vp,s.payload_volume)>0
    assert min(s.antibody_q,s.antibody_clearance,s.payload_clearance,s.permeability,s.metabolism)>=0
    assert s.half>0 and np.all(np.array(s.ki)>=0) and np.all(np.array(s.k0)>=0)
    for a in (s.recycling_rates,s.processing_rates,s.degradation_rates):
        assert not a or (len(a)==3 and np.all(np.array(a)>=0))
    assert all(bool(a)==bool(s.recycling_rates) for a in (s.processing_rates,s.degradation_rates))
    assert len(p.c50_uM)==len(p.maximum_rate_h)==5
    assert np.all(np.array(p.c50_uM)>0) and np.all(np.array(p.maximum_rate_h)>=0)

def integrate(theta,s=BIOLOGY,p=PD,tight=False,points=3,method='BDF'):
    check_inputs(s,p)
    y0=np.zeros(97)  # 87 material states + 5 cumulative hazards + 5 AUCs
    y0[7:87].reshape(5,16)[:,:2]=density(s)
    tt=[]; yy=[]
    times=np.unique(np.r_[np.linspace(0,s.horizon_h,points),s.infusion_h])
    for start,end,on in [(0,s.infusion_h,True),(s.infusion_h,s.horizon_h,False)]:
        material=make_rhs(theta,s,on)
        def rhs(t,y):
            d=np.zeros_like(y);d[:87]=material(t,y[:87])
            conc=y[7:87].reshape(5,16)[:,14]
            d[87:92]=np.array(p.maximum_rate_h)*conc/(np.array(p.c50_uM)+conc)
            d[92:97]=conc
            return d
        sol=solve_ivp(rhs,(start,end),y0,method=method,rtol=1e-8 if tight else 2e-6,
                      atol=1e-11 if tight else 2e-9,dense_output=True)
        if not sol.success:raise RuntimeError(sol.message)
        ts=times[(times>=start)&(times<=end)] if start==0 else times[(times>start)&(times<=end)]
        tt.extend(ts);yy.extend(sol.sol(ts).T);y0=sol.y[:,-1]
    y=np.array(yy);t=np.array(tt);z=y[:,7:87].reshape(-1,5,16)
    if y.min() < -2e-6:
        if not tight:return integrate(theta,s,p,True,points,method)
        if method=='BDF':return integrate(theta,s,p,True,points,'Radau')
        raise RuntimeError('Unresolved negative state')
    kill=-np.expm1(-y[:,87:92])
    cap=s.total_cells*np.array(s.fractions)*s.cell_area*1e-4
    vols=s.total_cells*np.array(s.fractions)*s.intracellular_volume
    ab=s.antibody_vc*(y[:,0]+y[:,2])+s.antibody_vp*(y[:,1]+y[:,3])+z[:,:,2:14].sum(axis=2)@cap+y[:,5]
    payload=s.antibody_vc*y[:,0]+s.antibody_vp*y[:,1]+(z[:,:,2:5]+z[:,:,8:11]).sum(axis=2)@cap+s.payload_volume*y[:,4]+z[:,:,14]@vols+y[:,6]
    dose=s.dose*np.minimum(t/s.infusion_h,1.)
    mass=max(abs(ab-dose).max(),abs(payload-dose).max())/max(s.dose,1.)
    if mass>2e-6:raise RuntimeError(f'Mass error exceeds tolerance: {mass}')
    return dict(t=t,y=y,z=z,kill=kill,healthy=kill[:,1:]@np.array(s.fractions[1:]),mass=mass)

def assess(theta,tight=False):
    rows=[]
    for name,s,p in INPUT_SET:
        r=integrate(theta,s,p,tight=tight); k=r['kill'][-1]
        rows.append([k[0],r['healthy'][-1],*k[1:]])
    a=np.array(rows)
    # Every constraint applies to every listed input variant, not an average variant.
    margins=np.r_[GLOBAL_LIMIT-a[:,1],(GROUP_LIMITS-a[:,2:]).ravel()]
    return dict(tumour=100*a[:,0].min(),healthy=100*a[:,1].max(),
                margins=100*margins,outcomes=100*a)


## 2. Search domains and interpretation


| Domain | Association rate, each arm (M⁻¹ s⁻¹) | Dissociation rate, each arm (s⁻¹) | Interpretation |
|---|---:|---:|---|
| Original | 100–10,000,000 | 0.00001–100 | V5 search limits |
| Expanded | 1–10,000,000 | 0.00001–1,000 | Numerical diagnostic; no claim of experimental attainability |
| Experimental reference | 120,000–560,000 | 0.00018–0.036 | Envelope from Mazor et al., Table 2; a different antibody/target system |

Reference: Mazor et al. (2015), *Insights into the molecular basis of a bispecific
antibody's target selectivity*, https://doi.org/10.1080/19420862.2015.1022695,
https://pmc.ncbi.nlm.nih.gov/articles/PMC4622944/ . Table 2 reports Octet kinetic
measurements for anti-CD4/anti-CD70 IgGs and DuetMabs. The limits above are the
smallest and largest reported association and dissociation rates. Applying this
envelope to both generic arms is an illustrative transfer, not a reconstruction
of that experiment. The rectangle also includes rate pairs that were not measured.
These limits must be replaced by target-specific measurements for actual design.
The original paper's optimised rates (for example, association 10⁴ M⁻¹ s⁻¹ and
dissociation 1 s⁻¹ in Table 2) also lie outside this particular experimental
envelope. Failure inside the envelope cannot establish that the paper's or V5's
rates are universally unattainable.

Each domain receives three seeds (8407, 2026, 11939), population 64, and at least
1,024 NSGA-II evaluations per seed, with observations at 256, 512 and 1,024.
For original and expanded domains, seed 8407 includes eight V5 starting points;
the other two seeds start independently from Latin hypercube samples. The
experimental-reference runs all start independently. Warm-start agreement is not
independent confirmation of V5.

The highest-tumour candidate from each feasible run is then refined. An auxiliary
tumour-kill threshold is maximised while every input variant separately exceeds
that threshold and satisfies the original healthy-cell constraints. This avoids
differentiating the minimum over variants and preserves V5's maximum-minimum
objective. Local refinement only examines this endpoint; it does not prove that
the entire trade-off surface has converged.

New representative trade-offs and endpoints are reintegrated at tighter ODE
tolerances. Numerical failures remain visible in the search records. No feasible
candidate found is not a mathematical proof of infeasibility.

## Interpretation

Boundary proximity means within 0.03 log10 units (about a 7% rate ratio) of an
allowed limit. The original count includes many almost-identical local iterates;
it is not a percentage of biological design space or a probability.
A boundary optimum is not inherently incorrect. It can be the appropriate design
if the boundary is a real experimental constraint. Here, the generic exploratory
limits are not established physical constraints, so their influence must be shown.

For readable output, retain one candidate per outcome bin of 0.1 percentage point
tumour killing, 0.002 overall healthy killing, and 0.05 subgroup killing, plus each
run's best tumour endpoint. Binning only reduces reporting density; it does not
define experimental equivalence. Reported fronts are sampled representative sets.

Convergence is examined using best feasible tumour killing and dominated objective
volume, with a fixed reference of 0% tumour killing, 0.95% overall healthy killing,
and 10% subgroup killing. The latter measures coverage of all three objectives;
it is not an efficacy or safety score. Checkpoint calculations use screening
tolerances; final reported candidates use tighter integration.

The inherited surface-binding relation couples association-rate changes to
dissociation from the doubly bound complex. With equal surface association rates,
q1 = koff1 sqrt(kon2/kon1) and q2 = koff2 sqrt(kon1/kon2). A tenfold decrease of
kon2 therefore multiplies q1 by 1/sqrt(10) and q2 by sqrt(10). Wolfram Alpha
confirmed the first square-root simplification in this review. Its Wolfram
Language evaluator was unavailable (internal service errors). No change to this
binding assumption is made here.

All reported combinations must keep the four rates together. Independently mixing
their minima and maxima does not preserve the calculated tumour selectivity.


In [ ]:
REUSE_SAVED_RUNS=True
from scipy.stats import qmc
from joblib import Parallel, delayed, parallel_config
import os

ROOT=Path.cwd().parent if Path.cwd().name=='repo' else Path.cwd()
OUT=ROOT/'repo/results_boundary_review'
DOMAINS={
 'original':np.array([(2,7),(-5,2),(2,7),(-5,2)],float),
 'expanded':np.array([(0,7),(-5,3),(0,7),(-5,3)],float),
 'experimental_reference':np.log10(np.array([(1.2e5,5.6e5),(1.8e-4,.036)]*2)),
}
SEEDS=(8407,2026,11939)

def evaluate(x,tight=False):
    a=assess(x,tight=tight)
    return dict(theta=list(map(float,x)),tumour=float(a['tumour']),healthy=float(a['healthy']),
        subgroup=float(a['outcomes'][:,2:].max()),minimum_margin=float(a['margins'].min()),
        outcomes=a['outcomes'].tolist())

def front(rows):
    if not rows:return []
    v=np.array([[-r['tumour'],r['healthy'],r['subgroup']] for r in rows])
    return [r for i,r in enumerate(rows) if not (np.all(v<=v[i]+1e-8,axis=1)&np.any(v<v[i]-1e-8,axis=1)).any()]

def run(domain,seed):
    path=OUT/f'{domain}_{seed}.json'
    if REUSE_SAVED_RUNS and path.exists() and json.loads(path.read_text()).get('complete'):return json.loads(path.read_text())
    random.seed(seed);bounds=DOMAINS[domain];history=[];cache={};checkpoints=[]
    def ev(x):
        key=tuple(map(float,x))
        if key not in cache:
            try:r=evaluate(x)
            except (RuntimeError,ValueError,FloatingPointError) as e:
                r=dict(theta=list(key),tumour=0.,healthy=100.,subgroup=100.,minimum_margin=-100.,outcomes=[],error=str(e))
            r['evaluation']=len(history)+1;history.append(r);cache[key]=r
        return cache[key]
    p=Problem(4,3,1);p.types[:]=[Real(*b) for b in bounds];p.constraints[:]='<=0'
    def objective(x):
        r=ev(x);return [-r['tumour'],r['healthy'],r['subgroup']],[-r['minimum_margin']]
    p.function=objective
    grid=qmc.scale(qmc.LatinHypercube(4,seed=seed).random(64),bounds[:,0],bounds[:,1])
    # Two independent random populations; one separately labelled warm-start run.
    warm=seed==8407 and domain!='experimental_reference'
    if warm:
        old=sorted(json.loads((ROOT/'repo/results_v5/observed_pareto_designs.json').read_text()),key=lambda r:r['tumour'])
        grid[:8]=[old[i]['theta'] for i in np.linspace(0,len(old)-1,8,dtype=int)]
    starts=[]
    for x in grid:
        s=Solution(p);s.variables[:]=x;starts.append(s)
    alg=NSGAII(p,population_size=64,generator=InjectedPopulation(starts))
    started=time.time()
    def save(complete=False,refinement=None):
        result=dict(domain=domain,seed=seed,warm_start=warm,bounds=bounds.tolist(),complete=complete,
            seconds=time.time()-started,nfe=alg.nfe,checkpoints=checkpoints,history=history,refinement=refinement)
        temp=path.with_suffix('.tmp');temp.write_text(json.dumps(result));temp.replace(path)
        return result
    for budget in (256,512,1024):
        while alg.nfe<budget:alg.step()
        feasible=[r for r in history if r['minimum_margin']>=0]
        checkpoints.append(dict(budget=budget,nfe=alg.nfe,distinct_evaluations=len(history),
            feasible=len(feasible),best_tumour=max((r['tumour'] for r in feasible),default=None),
            least_violation=max(r['minimum_margin'] for r in history)))
        save();print(domain,seed,checkpoints[-1],flush=True)
    # Smooth max-min formulation: optimise an auxiliary tumour-kill threshold,
    # constraining each variant separately rather than differentiating min/max.
    feasible=[r for r in history if r['minimum_margin']>=0]
    refinement=None
    if feasible:
        best=max(feasible,key=lambda r:r['tumour']);x0=np.r_[best['theta'],best['tumour']]
        def constraints(z):
            r=ev(z[:4]);a=np.array(r['outcomes'])
            if a.shape!=(3,6):return np.full(18,-100.)
            return np.r_[a[:,0]-z[4],100*GLOBAL_LIMIT-a[:,1],(100*GROUP_LIMITS-a[:,2:]).ravel()]-2e-4
        fit=minimize(lambda z:-z[4],x0,method='SLSQP',bounds=[tuple(b) for b in bounds]+[(0,100)],
            constraints=[dict(type='ineq',fun=constraints)],options=dict(maxiter=60,ftol=1e-7,eps=1e-3))
        ev(fit.x[:4]);refinement=dict(success=bool(fit.success),message=str(fit.message),nit=int(fit.nit))
    result=save(True,refinement);print('COMPLETE',domain,seed,len(history),flush=True);return result


# Cached outputs are valid only for the same declared study inputs.
current_configuration={
    'biological_inputs':[{'name':n,'biology':asdict(s),'kill_model':asdict(p)} for n,s,p in INPUT_SET],
    'bounds_log10':{k:v.tolist() for k,v in DOMAINS.items()},
    'overall_healthy_limit':GLOBAL_LIMIT,'subgroup_limits':GROUP_LIMITS.tolist(),
    'seeds':list(SEEDS),'population_size':64,'global_evaluations':1024,
    'local_iteration_limit':60,'reporting_bins_percentage_points':[.1,.002,.05],
    'boundary_distance_log10':.03,
}
current_configuration=json.loads(json.dumps(current_configuration))
configuration_path=OUT/'study_configuration.json'
if REUSE_SAVED_RUNS:
    if not configuration_path.exists():
        raise RuntimeError('No saved study configuration. Set REUSE_SAVED_RUNS=False to run the study.')
    recorded=json.loads(configuration_path.read_text())
    changed=[key for key,value in current_configuration.items() if recorded.get(key)!=value]
    if changed:
        raise RuntimeError('Inputs differ from saved results: '+', '.join(changed)+'. Set REUSE_SAVED_RUNS=False.')
else:
    OUT.mkdir(exist_ok=True,parents=True)
    configuration_path.write_text(json.dumps(current_configuration,indent=2))
# Any edit to equations or algorithms also requires REUSE_SAVED_RUNS=False.


## 3. Reassess the original boundary count
The count of search iterates is not a fraction of the continuous design space. Compare raw entries with one representative per declared outcome bin.

In [ ]:
from platypus import Hypervolume
import hashlib

def hypervolume(rows):
    if not rows:return 0.
    p=Problem(4,3);solutions=[]
    for r in front(rows):
        s=Solution(p);s.objectives[:]=[-r['tumour'],r['healthy'],r['subgroup']];solutions.append(s)
    return float(Hypervolume(minimum=[-100,0,0],maximum=[0,.95,10]).calculate(solutions))

def boundary(rows,bounds):
    if not rows:return dict(count=0,total=0,percent=None,by_rate={})
    x=np.array([r['theta'] for r in rows]);near=(x-bounds[:,0]<=.03)|(bounds[:,1]-x<=.03)
    return dict(count=int(near.any(axis=1).sum()),total=len(rows),percent=float(100*near.any(axis=1).mean()),
        by_rate={n:dict(lower=int((x[:,j]-bounds[j,0]<=.03).sum()),upper=int((bounds[j,1]-x[:,j]<=.03).sum())) for j,n in enumerate(RATES)})

def representatives(rows):
    # Declared output resolution in percentage points, not a biological equivalence claim.
    bins={}
    for r in sorted(rows,key=lambda r:(-r['tumour'],r['healthy'],r['subgroup'])):
        key=tuple(np.floor(np.array([r['tumour'],r['healthy'],r['subgroup']])/[.1,.002,.05]).astype(int))
        if key not in bins:bins[key]=r
    return list(bins.values())

def analyse_results(reuse_preverified=False):
    runs=[json.loads((OUT/f'{d}_{s}.json').read_text()) for d in DOMAINS for s in SEEDS]
    assert all(r['complete'] for r in runs)
    old=json.loads((ROOT/'repo/results_v5/observed_pareto_designs.json').read_text())
    candidates={};metadata=[];screened_pools={};domain_keys={}
    for run in runs:
        feasible=[r for r in run['history'] if r['minimum_margin']>=-1e-5 and not r.get('error')]
        selected=[]
        # Always retain endpoint and least-violating candidate independently of output binning.
        selected+=sorted(feasible,key=lambda r:-r['tumour'])[:1]
        selected+=[max(run['history'],key=lambda r:r['minimum_margin'])]
        screened_pools.setdefault(run['domain'],[]).extend(feasible)
        for r in selected:candidates[tuple(r['theta'])]=r
        metadata.append(dict(domain=run['domain'],seed=run['seed'],warm_start=run['warm_start'],
             selected_keys=[tuple(r['theta']) for r in selected],screened_feasible=len(feasible),
             errors=sum(bool(r.get('error')) for r in run['history']),evaluations=len(run['history']),
             refinement=run['refinement'],screened_hypervolume=hypervolume(feasible)))
    # Pool independent searches before expensive reintegration; retain every run's
    # endpoint separately. The final exported front is a representative archive.
    for domain,rows in screened_pools.items():
        unique=list({tuple(r['theta']):r for r in rows}.values())
        selected=representatives(front(unique))
        domain_keys[domain]=[tuple(r['theta']) for r in selected]
        for r in selected:candidates[tuple(r['theta'])]=r
    print('Tight reintegration of representative fronts and endpoints:',len(candidates),flush=True)
    tight=[]
    if reuse_preverified and (OUT/'preverified.json').exists():
        tight=[r for r in json.loads((OUT/'preverified.json').read_text()) if tuple(r['theta']) in candidates]
    completed={tuple(r['theta']) for r in tight}
    print('Already tightly integrated:',len(completed),flush=True)
    with parallel_config(backend='loky',inner_max_num_threads=1):
        for row in Parallel(n_jobs=6,return_as='generator_unordered')(delayed(evaluate)(x,True) for x in candidates if x not in completed):
            tight.append(row)
            if len(tight)%100==0:print('Tightly integrated',len(tight),'of',len(candidates),flush=True)
    lookup={tuple(r['theta']):r for r in tight}
    (OUT/'tight_evaluations.json').write_text(json.dumps(tight))
    checkpoints=[];perrun=[];pooled={}
    for run,meta in zip(runs,metadata):
        rr=[lookup[k] for k in meta.pop('selected_keys')];feasible=[r for r in rr if r['minimum_margin']>=-1e-7]
        best=max(feasible,key=lambda r:r['tumour']) if feasible else None
        perrun.append(dict(**meta,best_tumour=best['tumour'] if best else None,
             endpoint=best,least_violation=max(r['minimum_margin'] for r in rr),
             least_violating_candidate=max(rr,key=lambda r:r['minimum_margin']),
             endpoint_boundary=boundary(feasible[:1],DOMAINS[run['domain']])))
        pooled.setdefault(run['domain'],[]).extend(feasible)
        for cp in run['checkpoints']:
            rows=[r for r in run['history'][:cp['distinct_evaluations']] if r['minimum_margin']>=0]
            checkpoints.append(dict(domain=run['domain'],seed=run['seed'],**cp,hypervolume=hypervolume(rows)))
    summary={}
    for domain,keys in domain_keys.items():
        pooled.setdefault(domain,[]).extend(lookup[k] for k in keys if lookup[k]['minimum_margin']>=-1e-7)
    for domain,rows in pooled.items():
        unique=list({tuple(r['theta']):r for r in rows}.values());pf=front(unique)
        best=max(pf,key=lambda r:r['tumour']) if pf else None
        summary[domain]=dict(best=best,representatives=len(pf),boundary=boundary(pf,DOMAINS[domain]),
             hypervolume=hypervolume(pf),cold_seed_best=[r['best_tumour'] for r in perrun if r['domain']==domain and not r['warm_start']])
        (OUT/f'{domain}_representative_front.json').write_text(json.dumps(pf))
    summary['v5']=dict(best=max(old,key=lambda r:r['tumour']),boundary=boundary(old,DOMAINS['original']),
        representative_boundary=boundary(representatives(old),DOMAINS['original']),hypervolume=hypervolume(old))
    summary['notes']=['All reported new candidates tightly reintegrated. Fronts are representative sampled sets, not complete or proven global optima.',
        'One warm-start and two cold-start runs for original/expanded; three cold runs for experimental reference.',
        'Experimental reference bounds are not target-specific feasibility evidence. No feasible result is not a proof of infeasibility.',
        'Hypervolume is normalised using fixed worst reference (tumour=0%, overall healthy=.95%, subgroup=10%). Checkpoint values use screening tolerances.',
        'Representative bin widths: .1 percentage point tumour, .002 overall healthy, .05 subgroup; binning is for reporting only.']
    (OUT/'summary.json').write_text(json.dumps(summary,indent=2));(OUT/'runs_summary.json').write_text(json.dumps(perrun,indent=2))
    pd.DataFrame(checkpoints).to_csv(OUT/'convergence.csv',index=False)
    table=[]
    for domain in DOMAINS:
        for r in json.loads((OUT/f'{domain}_representative_front.json').read_text()):
            rates=10**np.array(r['theta']);table.append(dict(domain=domain,**dict(zip(RATES,rates)),KD1_uM=1e6*rates[1]/rates[0],KD2_uM=1e6*rates[3]/rates[2],tumour_percent=r['tumour'],healthy_percent=r['healthy'],subgroup_percent=r['subgroup']))
    pd.DataFrame(table).to_csv(OUT/'binding_combinations.csv',index=False)
    print(json.dumps({d:{k:v for k,v in s.items() if k!='best'} for d,s in summary.items() if isinstance(s,dict)},indent=2),flush=True)


old=json.loads((ROOT/'repo/results_v5/observed_pareto_designs.json').read_text())
display(pd.DataFrame([{'Set':'All V5 entries',**{k:v for k,v in boundary(old,DOMAINS['original']).items() if k!='by_rate'}},{'Set':'V5 outcome representatives',**{k:v for k,v in boundary(representatives(old),DOMAINS['original']).items() if k!='by_rate'}}]))

## 4. Longer independent searches
Three seeds per domain; at least 1,024 global-search evaluations per seed. One original and one expanded run use V5 starting points; remaining runs start independently. Final endpoint refinement enforces each input variant separately.

In [ ]:
with parallel_config(backend='loky',inner_max_num_threads=1):
    runs=Parallel(n_jobs=6)(delayed(run)(d,s) for d in DOMAINS for s in SEEDS)
assert all(r['complete'] for r in runs)
if not REUSE_SAVED_RUNS or not (OUT/'summary.json').exists():
    analyse_results()
print('Completed runs:',len(runs))

## 5. Results
Checkpoint curves precede endpoint refinement. Final candidate plots use tighter ODE integration. The experimental-reference domain may contain no candidate meeting the unchanged requirements; that is a finding within this transferred reference domain, not a universal statement about antibody engineering.

In [ ]:

def endpoint_checks():
    summary=json.loads((OUT/'summary.json').read_text());rows=[]
    for domain in ['original','expanded']:
        theta=summary[domain]['best']['theta']
        for name,biology,pd_model in INPUT_SET:
            a=integrate(theta,biology,pd_model,tight=True,points=97)
            b=integrate(theta,biology,pd_model,tight=True,points=97,method='Radau')
            delta=float(100*np.max(abs(a['kill']-b['kill'])))
            assert delta<.001
            rows.append(dict(domain=domain,input=name,maximum_kill_difference_pp=delta,
                relative_mass_error=a['mass'],minimum_sampled_state=float(a['y'].min())))
    pd.DataFrame(rows).to_csv(OUT/'endpoint_checks.csv',index=False)
    print(pd.DataFrame(rows).to_string(index=False))


def plot_results():
    plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
    summary=json.loads((OUT/'summary.json').read_text());runs=json.loads((OUT/'runs_summary.json').read_text())
    convergence=pd.read_csv(OUT/'convergence.csv')
    colors={'original':'#2368a0','expanded':'#b84e24','experimental_reference':'#497942'}
    titles={'original':'Original limits','expanded':'Expanded limits','experimental_reference':'Experimental reference'}
    fig,axs=plt.subplots(1,2,figsize=(10.5,4.2),layout='constrained')
    old=json.loads((ROOT/'repo/results_v5/observed_pareto_designs.json').read_text())
    for ax,key,label in zip(axs,['healthy','subgroup'],['Overall healthy cells killed (%)','Most affected healthy subgroup killed (%)']):
        ax.scatter([r[key] for r in old],[r['tumour'] for r in old],s=9,color='.75',label='V5')
        for d in DOMAINS:
            rows=json.loads((OUT/f'{d}_representative_front.json').read_text())
            if rows:ax.scatter([r[key] for r in rows],[r['tumour'] for r in rows],s=17,alpha=.65,color=colors[d],label=titles[d])
        ax.set(xlabel=label,ylabel='Lowest tumour cell kill across inputs (%)');ax.grid(alpha=.2)
    axs[0].legend(fontsize=8);fig.suptitle('Trade-offs after longer searches: tightly integrated representatives')
    fig.savefig(OUT/'tradeoffs.png');plt.close(fig)
    fig,axs=plt.subplots(1,2,figsize=(10.5,4.2),layout='constrained')
    for d in DOMAINS:
        for seed in SEEDS:
            s=convergence[(convergence.domain==d)&(convergence.seed==seed)]
            for ax,col in zip(axs,['best_tumour','hypervolume']):
                ax.plot(s.budget,s[col],marker='o',color=colors[d],alpha=.8,ls='--' if seed==8407 else '-',label=titles[d] if seed==2026 else None)
    axs[0].axhline(summary['v5']['best']['tumour'],color='.4',lw=1,ls=':',label='V5 best')
    axs[0].set(ylabel='Best feasible tumour cell kill (%)',xlabel='Global-search evaluations per run')
    axs[1].set(ylabel='Normalised dominated objective volume',xlabel='Global-search evaluations per run')
    axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
    for ax in axs:ax.grid(alpha=.2)
    fig.suptitle('Search progress before local refinement (screening tolerances)')
    fig.savefig(OUT/'convergence.png');plt.close(fig)
    fig,axs=plt.subplots(1,2,figsize=(10.5,4.2),layout='constrained')
    for d in DOMAINS:
        rows=json.loads((OUT/f'{d}_representative_front.json').read_text())
        if not rows:continue
        a=10**np.array([r['theta'] for r in rows]);t=[r['tumour'] for r in rows]
        axs[0].scatter(a[:,2],t,s=18,alpha=.65,color=colors[d],label=titles[d])
        axs[1].scatter(a[:,1]/a[:,0]*1e6,a[:,3]/a[:,2]*1e6,s=18,alpha=.65,color=colors[d],label=titles[d])
    axs[0].axvline(100,color='.4',ls=':',label='Original lower limit')
    axs[0].set(xscale='log',xlabel='Target 2 association rate (M⁻¹ s⁻¹)',ylabel='Lowest tumour cell kill (%)')
    axs[1].set(xscale='log',yscale='log',xlabel='Target 1 dissociation constant (µM)',ylabel='Target 2 dissociation constant (µM)')
    for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.suptitle('Complete rate combinations remain linked; points are numerical candidates')
    fig.savefig(OUT/'binding_dependence.png');plt.close(fig)
    # A compact table preserves the independent-run information behind pooled results.
    rows=[]
    for r in runs:
        rows.append({'Domain':titles[r['domain']],'Seed':r['seed'],'V5 starts':r['warm_start'],
            'Evaluations':r['evaluations'],'Best tumour (%)':r['best_tumour'],
            'Least constraint margin (pp)':r['least_violation'],'Numerical failures':r['errors'],
            'Local termination':r['refinement']['message'] if r['refinement'] else 'No feasible start'})
    pd.DataFrame(rows).to_csv(OUT/'run_comparison.csv',index=False)
    print(pd.DataFrame(rows).to_string(index=False))


if not REUSE_SAVED_RUNS or not (OUT/'endpoint_checks.csv').exists():
    endpoint_checks()
plot_results()
from IPython.display import Image
for name in ['tradeoffs','convergence','binding_dependence']:
    display(Image(filename=str(OUT/(name+'.png'))))
display(pd.read_csv(OUT/'run_comparison.csv'))